In [1]:
!pip install -q ijson
import os, glob, random, time
import ijson
from huggingface_hub import snapshot_download
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import AutoTokenizer
 
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None
 
VOCAB_SIZE = 50_257              # YOUR experiment knob (e.g. 16k / 32k / 50k)
CHARS_PER_FILE = 100_000_000      # ~100M characters of text sampled from each source
KEEP_PROB = 0.3                   # sample documents across the whole file, not just the start
SEED = 42
OUT_DIR = "/kaggle/working/legal_bpe"
os.makedirs(OUT_DIR, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 3.4 MB/s eta 0:00:00a 0:00:01


In [2]:
DATA_DIR = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["*Phase_1*", "*Phase1*"], local_dir="/tmp/cs781_data")
all_json = glob.glob(f"{DATA_DIR}/**/*.json", recursive=True) + glob.glob("/kaggle/input/**/*.json", recursive=True)
NAMES = ["train_acts.json", "train_high_courts.json", "train_supreme_court.json", "train_tribunals.json"]
FILES = [next((f for f in all_json if os.path.basename(f) == n), None) for n in NAMES]
FILES = [f for f in FILES if f]
print("Training files:", FILES)
 
def iter_texts(path):
    with open(path, "rb") as f:
        first = f.read(1)
        while first.isspace():
            first = f.read(1)
        f.seek(0)
        it = ijson.items(f, "item") if first == b"[" else (v for _, v in ijson.kvitems(f, ""))
        for rec in it:
            t = rec.get("text") if isinstance(rec, dict) else rec
            if isinstance(t, str) and t:
                yield t
 
def sampled_corpus():
    rng = random.Random(SEED)
    for path in FILES:
        n_chars = 0
        for text in iter_texts(path):
            if rng.random() > KEEP_PROB:
                continue
            yield text
            n_chars += len(text)
            if n_chars >= CHARS_PER_FILE:
                break
        print(f"  {os.path.basename(path)}: {n_chars / 1e6:.0f}M chars used")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Training files: ['/tmp/cs781_data/Phase_1_Pretraining/Train/train_acts.json', '/tmp/cs781_data/Phase_1_Pretraining/Train/train_high_courts.json', '/tmp/cs781_data/Phase_1_Pretraining/Train/train_supreme_court.json', '/tmp/cs781_data/Phase_1_Pretraining/Train/train_tribunals.json']


In [5]:
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)   
tok.decoder = decoders.ByteLevel()                                     
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=["<|endoftext|>"],                  
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=True,
)
t0 = time.time()
tok.train_from_iterator(sampled_corpus(), trainer=trainer)
tok.save(os.path.join(OUT_DIR, "tokenizer.json"))
print(f"Saved {OUT_DIR}/tokenizer.json | vocab {tok.get_vocab_size()} | {time.time() - t0:.0f}s")

  train_acts.json: 100M chars used
  train_high_courts.json: 100M chars used
  train_supreme_court.json: 100M chars used
  train_tribunals.json: 100M chars used



Saved /kaggle/working/legal_bpe/tokenizer.json | vocab 50257 | 102s


In [6]:
gpt2 = AutoTokenizer.from_pretrained("gpt2")
rng = random.Random(123)
sample = []
for path in FILES:
    for i, text in enumerate(iter_texts(path)):
        if i > 5000: break
        if rng.random() < 0.01: sample.append(text[:5000])
words = sum(len(t.split()) for t in sample)
ours = sum(len(tok.encode(t).ids) for t in sample)
theirs = sum(len(gpt2.encode(t)) for t in sample)
print(f"Tokens per word  | ours: {ours / words:.3f} | GPT-2: {theirs / words:.3f}")
 
s = "The appellant filed a writ petition under Article 226 of the Constitution of India."
print("ours :", tok.encode(s).tokens)
print("gpt2 :", gpt2.tokenize(s))
print("round-trip OK:", tok.decode(tok.encode(s).ids) == s)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1091 > 1024). Running this sequence through the model will result in indexing errors


Tokens per word  | ours: 1.314 | GPT-2: 1.431
ours : ['The', 'Ġappellant', 'Ġfiled', 'Ġa', 'Ġwrit', 'Ġpetition', 'Ġunder', 'ĠArticle', 'Ġ226', 'Ġof', 'Ġthe', 'ĠConstitution', 'Ġof', 'ĠIndia', '.']
gpt2 : ['The', 'Ġappellant', 'Ġfiled', 'Ġa', 'Ġwrit', 'Ġpetition', 'Ġunder', 'ĠArticle', 'Ġ226', 'Ġof', 'Ġthe', 'ĠConstitution', 'Ġof', 'ĠIndia', '.']
round-trip OK: True
